In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from pathlib import Path
import scanpy.external as sce

In [2]:
def _toarray(X):
    return X.toarray() if sparse.issparse(X) else np.asarray(X)

def standardize_10x_multimodal(adata, hashtag_patterns=("Hashtag", "HTO")):
    """
    把 10x h5 (gex_only=False) 读入的混合矩阵拆分并整理为：
      - GEX 留在 adata.X（原始计数备份到 adata.layers["counts"]）
      - ADT 放到 adata.obsm["protein_counts"]；列信息记到 adata.uns["protein_vars"]
      - HTO 放到 adata.obsm["hashing_counts"]；同时把同样数据复制到 adata.obs 的多列
    返回一个字典，包含各模块的特征名，便于后续核对。
    """
    assert "feature_types" in adata.var.columns, "var['feature_types'] 缺失，确认是 10x h5 读取结果"

    # 1) 三类掩码
    is_ac = adata.var["feature_types"].astype(str) == "Antibody Capture"
    # HTO：根据 var_names 或 gene_ids 中是否匹配 “Hashtag/HTO”
    name_has_hashtag = adata.var_names.str.contains("|".join(hashtag_patterns), case=False, na=False)
    gid_has_hashtag  = adata.var.get("gene_ids", pd.Series("", index=adata.var_names))\
                          .astype(str).str.contains("|".join(hashtag_patterns), case=False, na=False)
    is_hto = is_ac & (name_has_hashtag | gid_has_hashtag)
    # ADT：Antibody Capture 但不是 HTO 的都视为蛋白
    is_adt = is_ac & (~is_hto)
    # GEX：其余为转录组
    is_gex = ~is_ac

    # 2) 取出子矩阵
    adata.layers["counts"] = adata.X.copy()  # 备份原始计数（混合的，后面会改掉）
    gex = adata[:, is_gex].copy()
    adt = adata[:, is_adt].copy()
    hto = adata[:, is_hto].copy()

    # 3) 把 GEX 留在 adata.X（只含 GEX）
    adata._inplace_subset_var(is_gex)        # 过滤到只剩 GEX 特征
    adata.layers["counts"] = adata.X.copy()  # 现在 layers["counts"]/X 都是 GEX 原始计数

    # 4) ADT → obsm/uns
    if adt.n_vars > 0:
        adata.obsm["protein_counts"] = _toarray(adt.X)
        # 记录列顺序和元信息，防止丢失
        adata.uns["protein_vars"] = pd.DataFrame(adt.var.copy())
        adata.uns["protein_var_names"] = adt.var_names.to_list()

    # 5) HTO → obsm + obs 多列（供 hashsolo/HTODemux 使用）
    if hto.n_vars > 0:
        H = _toarray(hto.X)
        adata.obsm["hashing_counts"] = H
        adata.uns["hashing_vars"] = pd.DataFrame(hto.var.copy())
        adata.uns["hashing_var_names"] = hto.var_names.to_list()

        # 同步到 obs（每个 HTO 一列）
        hto_df = pd.DataFrame(H, index=adata.obs_names, columns=hto.var_names)
        adata.obs = adata.obs.join(hto_df)

    return {
        "n_gex": int(gex.n_vars),
        "n_adt": int(adt.n_vars),
        "n_hto": int(hto.n_vars),
        "gex_features": gex.var_names.to_list(),
        "adt_features": adt.var_names.to_list(),
        "hto_features": hto.var_names.to_list(),
    }

In [3]:
def find_h5(sample_dir: Path) -> Path | None:
    h5 = sample_dir / "outs" / "filtered_feature_bc_matrix.h5"
    return h5 if h5.exists() else None

In [4]:
def process_sample(sample_dir: Path):
    h5 = find_h5(sample_dir)
    if h5 is None:
        print(f"[skip] {sample_dir.name}: no filtered_feature_bc_matrix.h5")
        return

    try:
        adata = sc.read_10x_h5(str(h5), gex_only=False)
    except Exception as e:
        print(f"[error] {sample_dir.name}: failed to read h5 -> {e}")
        return

    info = standardize_10x_multimodal(adata)
    hto_cols = info["hto_features"]

    # Run HashSolo if HTOs exist
    if len(hto_cols) > 0:
        try:
            sce.pp.hashsolo(
                adata,
                cell_hashing_columns=hto_cols,
                number_of_noise_barcodes=max(len(hto_cols) - 1, 0),
                inplace=True,
            )
            # Post-process classification
            if "Classification" in adata.obs.columns:
                cls = adata.obs["Classification"].astype(str)
                adata.obs["hash_tag"] = cls  # raw assignment
                adata.obs["hash_status"] = np.where(
                    cls.str.lower().isin(["doublet", "negative"]),
                    cls.str.lower(),
                    "singlet",
                )
            else:
                # Fallback if HashSolo didn't create it for some reason
                adata.obs["hash_tag"] = pd.NA
                adata.obs["hash_status"] = pd.NA
        except Exception as e:
            print(f"[warn] {sample_dir.name}: hashsolo failed -> {e}")
            # Still create empty columns to keep schema stable
            adata.obs["hash_tag"] = pd.NA
            adata.obs["hash_status"] = pd.NA
    else:
        # No HTOs: create empty columns for consistent downstream
        adata.obs["hash_tag"] = pd.NA
        adata.obs["hash_status"] = pd.NA

    # Save obs to outs/
    outs = sample_dir / "outs"
    outs.mkdir(parents=True, exist_ok=True)
    csv_path = outs / "obs.csv.gz"
    parquet_path = outs / "obs.parquet"

    try:
        adata.obs.to_csv(csv_path, index=True)  # index = barcodes
    except Exception as e:
        print(f"[warn] {sample_dir.name}: failed to write CSV -> {e}")

    try:
        # Parquet is faster/smaller for large obs tables
        adata.obs.to_parquet(parquet_path, index=True)
    except Exception as e:
        print(f"[warn] {sample_dir.name}: failed to write Parquet -> {e}")

    print(f"[ok] {sample_dir.name}: obs -> {csv_path.name}, {parquet_path.name}")


In [5]:
BASE = Path("/home/sduan/ia/alignments/count")

In [6]:
samples = sorted([p for p in BASE.iterdir() if p.is_dir()])

In [7]:
samples

[PosixPath('/home/sduan/ia/alignments/count/582C_CZI-IA9924320'),
 PosixPath('/home/sduan/ia/alignments/count/582C_CZI-IA9924321'),
 PosixPath('/home/sduan/ia/alignments/count/582C_CZI-IA9924322'),
 PosixPath('/home/sduan/ia/alignments/count/582C_CZI-IA9924323'),
 PosixPath('/home/sduan/ia/alignments/count/582C_CZI-IA9924324'),
 PosixPath('/home/sduan/ia/alignments/count/583B_CZI-IA9924325'),
 PosixPath('/home/sduan/ia/alignments/count/583B_CZI-IA9924326'),
 PosixPath('/home/sduan/ia/alignments/count/583B_CZI-IA9924327'),
 PosixPath('/home/sduan/ia/alignments/count/583B_CZI-IA9924328'),
 PosixPath('/home/sduan/ia/alignments/count/591C_CZI-IA10034920'),
 PosixPath('/home/sduan/ia/alignments/count/591C_CZI-IA10034923'),
 PosixPath('/home/sduan/ia/alignments/count/591C_CZI-IA10034924'),
 PosixPath('/home/sduan/ia/alignments/count/591C_CZI-IA10034926'),
 PosixPath('/home/sduan/ia/alignments/count/621B_CZI-IA10244331'),
 PosixPath('/home/sduan/ia/alignments/count/621B_CZI-IA10244332'),
 Pos

In [8]:
for s in samples:
    process_sample(s)

/home/sduan/miniforge3/envs/scverse/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/sduan/miniforge3/envs/scverse/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/sduan/miniforge3/envs/scverse/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/sduan/miniforge3/envs/scverse/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Please cite HashSolo paper:
https://www.cell.com/cell-systems/fulltext/S2405-4712(20)30195-2
[ok] 582C_CZI-IA9924320: obs -> obs.csv.gz, obs.parquet


/home/sduan/miniforge3/envs/scverse/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/sduan/miniforge3/envs/scverse/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/sduan/miniforge3/envs/scverse/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/sduan/miniforge3/envs/scverse/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Please cite HashSolo paper:
https://www.cell.com/cell-systems/fulltext/S2405-4712(20)30195-2
[ok] 582C_CZI-IA9924321: obs -> obs.csv.gz, obs.parquet


/home/sduan/miniforge3/envs/scverse/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/sduan/miniforge3/envs/scverse/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/sduan/miniforge3/envs/scverse/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/sduan/miniforge3/envs/scverse/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


...

Please cite HashSolo paper:
https://www.cell.com/cell-systems/fulltext/S2405-4712(20)30195-2
[ok] D543_CZINY-0585: obs -> obs.csv.gz, obs.parquet


/home/sduan/miniforge3/envs/scverse/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/sduan/miniforge3/envs/scverse/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/sduan/miniforge3/envs/scverse/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/sduan/miniforge3/envs/scverse/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Please cite HashSolo paper:
https://www.cell.com/cell-systems/fulltext/S2405-4712(20)30195-2
[ok] D564_CZINY-0649: obs -> obs.csv.gz, obs.parquet


/home/sduan/miniforge3/envs/scverse/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/sduan/miniforge3/envs/scverse/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/sduan/miniforge3/envs/scverse/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/sduan/miniforge3/envs/scverse/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Please cite HashSolo paper:
https://www.cell.com/cell-systems/fulltext/S2405-4712(20)30195-2
[ok] D570_CZINY-0713: obs -> obs.csv.gz, obs.parquet


In [9]:
1

1

In [71]:
adata = sc.read_10x_h5(
    "/home/sduan/ia/alignments/count/583B_CZI-IA9924325/outs/filtered_feature_bc_matrix.h5",
    gex_only=False
)

/home/sduan/miniforge3/envs/scverse/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [72]:
adata

AnnData object with n_obs × n_vars = 8031 × 55012
    var: 'gene_ids', 'feature_types', 'genome', 'pattern', 'read', 'sequence'

In [73]:
info = standardize_10x_multimodal(adata)

/home/sduan/miniforge3/envs/scverse/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/sduan/miniforge3/envs/scverse/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/sduan/miniforge3/envs/scverse/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [74]:
print({k: info[k] for k in ["n_gex", "n_adt", "n_hto"]})

{'n_gex': 54872, 'n_adt': 137, 'n_hto': 3}


In [75]:
import scanpy.external as sce
hto_cols = info["hto_features"]
if len(hto_cols) > 0:
    sce.pp.hashsolo(adata, cell_hashing_columns=hto_cols, number_of_noise_barcodes=len(hto_cols)-1)

Please cite HashSolo paper:
https://www.cell.com/cell-systems/fulltext/S2405-4712(20)30195-2


In [76]:
adata.obs["hash_tag"] = adata.obs["Classification"]                # 具体分到哪个 hashtag
adata.obs["hash_status"] = np.where(
    adata.obs["Classification"].isin(["Doublet", "Negative"]),     # 有些数据集会出现 "Negative"
    adata.obs["Classification"].str.lower(),
    "singlet"
)

In [77]:
# print(adata.obs["hash_status"].value_counts())
print(adata.obs["hash_tag"].value_counts())

hash_tag
583B-SPL-45    3879
583B-LLN-47    2362
Doublet         915
583B-LNG-46     875
Name: count, dtype: int64


In [80]:
adata

AnnData object with n_obs × n_vars = 8031 × 54872
    obs: '583B-SPL-45', '583B-LNG-46', '583B-LLN-47', 'most_likely_hypothesis', 'cluster_feature', 'negative_hypothesis_probability', 'singlet_hypothesis_probability', 'doublet_hypothesis_probability', 'Classification', 'hash_tag', 'hash_status'
    var: 'gene_ids', 'feature_types', 'genome', 'pattern', 'read', 'sequence'
    uns: 'protein_vars', 'protein_var_names', 'hashing_vars', 'hashing_var_names'
    obsm: 'protein_counts', 'hashing_counts'
    layers: 'counts'